In [1]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
import pickle
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score

c:\Projects\vlr-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
splits = joblib.load("splits/6_windows_all_features_knn.pkl")

In [3]:
splits[0]['X_train_raw'].shape

(1380, 1063)

In [4]:
from sklearn.impute import KNNImputer, SimpleImputer

feature_names = splits[0]["X_train_raw"].columns

for split in splits:

    # imputer = SimpleImputer(strategy="median")
    imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")

    split["X_train_raw"] = pd.DataFrame(imputer.fit_transform(split["X_train_raw"]), columns=feature_names)
    split["X_val_raw"] = pd.DataFrame(imputer.transform(split["X_val_raw"]), columns=feature_names)
    split["X_test_raw"] = pd.DataFrame(imputer.transform(split["X_test_raw"]), columns=feature_names)

    split["X_train_raw"] = split["X_train_raw"].loc[:, split["X_train_raw"].columns.str.startswith('delta_')]
    split["X_val_raw"] = split["X_val_raw"].loc[:, split["X_val_raw"].columns.str.startswith('delta_')]
    split["X_test_raw"] = split["X_test_raw"].loc[:, split["X_test_raw"].columns.str.startswith('delta_')]

In [5]:
splits[0]['X_train_raw'].shape

(1380, 354)

In [6]:
print(splits[0]['X_train_raw'].columns.to_list())

['delta_map_attack_rounds_sum', 'delta_map_attack_rounds_sum_2', 'delta_map_attack_rounds_sum_4', 'delta_map_attack_rounds_sum_6', 'delta_map_defense_rounds_sum', 'delta_map_defense_rounds_sum_2', 'delta_map_defense_rounds_sum_4', 'delta_map_defense_rounds_sum_6', 'delta_map_duration_sum', 'delta_map_duration_sum_2', 'delta_map_duration_sum_4', 'delta_map_duration_sum_6', 'delta_map_patch_played', 'delta_map_patch_score_delta_avg', 'delta_map_patch_score_delta_avg_2', 'delta_map_patch_score_delta_avg_4', 'delta_map_patch_score_delta_avg_6', 'delta_map_played', 'delta_map_score_delta_avg', 'delta_map_score_delta_avg_2', 'delta_map_score_delta_avg_4', 'delta_map_score_delta_avg_6', 'delta_map_score_delta_sum', 'delta_map_score_delta_sum_2', 'delta_map_score_delta_sum_4', 'delta_map_score_delta_sum_6', 'delta_map_specific_played', 'delta_map_specific_score_delta_avg', 'delta_map_specific_score_delta_avg_2', 'delta_map_specific_score_delta_avg_4', 'delta_map_specific_score_delta_avg_6', 'd

In [7]:
def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi

In [8]:
def make_objective(X_train, y_train, X_val, y_val):
    def objective(trial):
        C = trial.suggest_float('C', 1e-5, 10, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1)

        all_val_true, all_val_proba = [], []

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled   = scaler.transform(X_val)

        selector = VarianceThreshold(threshold=0.01)
        X_train_sel = selector.fit_transform(X_train_scaled)
        X_val_sel   = selector.transform(X_val_scaled)

        model = LogisticRegression(C=C, l1_ratio=l1_ratio, solver='saga', max_iter=5000)
        model.fit(X_train_sel, y_train)

        val_proba = model.predict_proba(X_val_sel)[:, 1]
        all_val_true.append(y_val); all_val_proba.append(val_proba)
        logloss = log_loss(all_val_true, all_val_proba)
        return logloss

    return objective

In [9]:
scores = []
parameters = []

for split in splits:
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    objective = make_objective(X_train, y_train, X_val, y_val)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=1000, n_jobs=-1)
    params = study.best_params
    parameters.append(params)
    print("Best parameters:", params)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel   = selector.transform(X_val_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_pred = model.predict(X_val_sel)
    scores.append(roc_auc_score(y_val, val_pred))

print(f"Scores: {scores}")
print(f"Parameters: {parameters}")
print(f"Mean logloss: {np.mean(scores)}")

[I 2026-07-22 21:18:08,198] A new study created in memory with name: no-name-50a3d19f-4b77-4440-96bb-ac513845776f
[I 2026-07-22 21:18:08,421] Trial 4 finished with value: 92.04502903688841 and parameters: {'C': 9.721838089354392e-05, 'l1_ratio': 0.686968815665116}. Best is trial 4 with value: 92.04502903688841.
[I 2026-07-22 21:18:08,433] Trial 3 finished with value: 95.07490639922918 and parameters: {'C': 1.7105575823999386e-05, 'l1_ratio': 0.5661929262990439}. Best is trial 4 with value: 92.04502903688841.
[I 2026-07-22 21:18:08,448] Trial 2 finished with value: 89.37384627223065 and parameters: {'C': 2.078984913250561e-05, 'l1_ratio': 0.5779468845449931}. Best is trial 2 with value: 89.37384627223065.
[I 2026-07-22 21:18:08,479] Trial 0 finished with value: 95.39038451925056 and parameters: {'C': 2.5210665358076778e-05, 'l1_ratio': 0.7201760905497437}. Best is trial 2 with value: 89.37384627223065.
[I 2026-07-22 21:18:08,482] Trial 6 finished with value: 93.1139564685286 and paramet

Best parameters: {'C': 0.006560361149082565, 'l1_ratio': 0.17562791301463665}


[I 2026-07-22 21:20:02,424] Trial 13 finished with value: 70.75091750500313 and parameters: {'C': 2.5440703961481003e-05, 'l1_ratio': 0.5916614118198833}. Best is trial 13 with value: 70.75091750500313.
[I 2026-07-22 21:20:02,434] Trial 4 finished with value: 68.78747969013703 and parameters: {'C': 3.2197726554010244e-05, 'l1_ratio': 0.49103295078012155}. Best is trial 4 with value: 68.78747969013703.
[I 2026-07-22 21:20:02,447] Trial 12 finished with value: 70.0453006736387 and parameters: {'C': 0.0007849033803762589, 'l1_ratio': 0.9945359419609894}. Best is trial 4 with value: 68.78747969013703.
[I 2026-07-22 21:20:02,530] Trial 5 finished with value: 70.5562609823736 and parameters: {'C': 0.0036974582794966114, 'l1_ratio': 0.966493547328892}. Best is trial 4 with value: 68.78747969013703.
[I 2026-07-22 21:20:02,552] Trial 6 finished with value: 70.07526275410481 and parameters: {'C': 0.0005040449286552218, 'l1_ratio': 0.07793744583877393}. Best is trial 4 with value: 68.787479690137

Best parameters: {'C': 6.673994439168969e-05, 'l1_ratio': 0.7717709571057689}


[I 2026-07-22 21:22:52,501] Trial 1 finished with value: 123.46356904954793 and parameters: {'C': 4.3424736187399985e-05, 'l1_ratio': 0.23308980381926936}. Best is trial 1 with value: 123.46356904954793.
[I 2026-07-22 21:22:52,517] Trial 12 finished with value: 126.03731029485482 and parameters: {'C': 1.4784648236702604e-05, 'l1_ratio': 0.8095214663790852}. Best is trial 1 with value: 123.46356904954793.
[I 2026-07-22 21:22:52,534] Trial 11 finished with value: 125.7600307799176 and parameters: {'C': 2.2804695950093484e-05, 'l1_ratio': 0.8017344249069893}. Best is trial 1 with value: 123.46356904954793.
[I 2026-07-22 21:22:52,537] Trial 3 finished with value: 128.3905892326528 and parameters: {'C': 0.0005020046159963528, 'l1_ratio': 0.4496124477535295}. Best is trial 1 with value: 123.46356904954793.
[I 2026-07-22 21:22:52,539] Trial 7 finished with value: 128.428868820985 and parameters: {'C': 0.0009136185830314897, 'l1_ratio': 0.8191871825994594}. Best is trial 1 with value: 123.4635

...

Best parameters: {'C': 0.027266090736686673, 'l1_ratio': 0.6841345292713716}


[I 2026-07-22 21:33:14,984] A new study created in memory with name: no-name-83da10ec-69fe-45db-97fe-036268e9a123
[I 2026-07-22 21:33:15,285] Trial 2 finished with value: 136.97313938527586 and parameters: {'C': 0.00011344391834860143, 'l1_ratio': 0.8572289459998915}. Best is trial 2 with value: 136.97313938527586.
[I 2026-07-22 21:33:15,317] Trial 4 finished with value: 131.80900776990853 and parameters: {'C': 0.00021597674978273784, 'l1_ratio': 0.8847043498156797}. Best is trial 4 with value: 131.80900776990853.
[I 2026-07-22 21:33:15,359] Trial 15 finished with value: 133.60356776984565 and parameters: {'C': 1.211571096119338e-05, 'l1_ratio': 0.3038980484799374}. Best is trial 4 with value: 131.80900776990853.
[I 2026-07-22 21:33:15,378] Trial 5 finished with value: 134.9180918202954 and parameters: {'C': 7.85480950680854e-05, 'l1_ratio': 0.18769115939360193}. Best is trial 4 with value: 131.80900776990853.
[I 2026-07-22 21:33:15,601] Trial 13 finished with value: 122.63260614784755

Best parameters: {'C': 0.004759455723732093, 'l1_ratio': 0.08698395236781287}


[I 2026-07-22 21:38:06,029] A new study created in memory with name: no-name-95833859-7440-4b26-bed2-9014debf33e8
[I 2026-07-22 21:38:06,474] Trial 3 finished with value: 137.66233073144372 and parameters: {'C': 3.9608381125274546e-05, 'l1_ratio': 0.7988385575952136}. Best is trial 3 with value: 137.66233073144372.
[I 2026-07-22 21:38:06,491] Trial 12 finished with value: 135.41028410879582 and parameters: {'C': 1.4709538169560667e-05, 'l1_ratio': 0.40497089567808664}. Best is trial 12 with value: 135.41028410879582.
[I 2026-07-22 21:38:06,588] Trial 13 finished with value: 138.12404712662507 and parameters: {'C': 0.0001632014754968603, 'l1_ratio': 0.16283880890073166}. Best is trial 12 with value: 135.41028410879582.
[I 2026-07-22 21:38:06,602] Trial 5 finished with value: 137.86766079092675 and parameters: {'C': 0.0012094689139093829, 'l1_ratio': 0.7963848753423531}. Best is trial 12 with value: 135.41028410879582.
[I 2026-07-22 21:38:06,649] Trial 16 finished with value: 139.0009387

Best parameters: {'C': 0.00219306123200391, 'l1_ratio': 0.2854144901029786}
Scores: [0.6344901888868513, 0.5, 0.6564335823874979, 0.6851063829787234, 0.6956606942889138, 0.6484727160735784]
Parameters: [{'C': 0.006560361149082565, 'l1_ratio': 0.17562791301463665}, {'C': 6.673994439168969e-05, 'l1_ratio': 0.7717709571057689}, {'C': 0.009647002447870038, 'l1_ratio': 0.6520235403924961}, {'C': 0.027266090736686673, 'l1_ratio': 0.6841345292713716}, {'C': 0.004759455723732093, 'l1_ratio': 0.08698395236781287}, {'C': 0.00219306123200391, 'l1_ratio': 0.2854144901029786}]
Mean logloss: 0.6366939274359273


In [10]:
window_results = []
all_val_true, all_val_proba = [], []
all_test_true, all_test_proba = [], []

for i, (split, params) in enumerate(zip(splits, parameters)):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel = selector.transform(X_val_scaled)
    X_test_sel = selector.transform(X_test_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_proba = model.predict_proba(X_val_sel)[:, 1]
    test_proba = model.predict_proba(X_test_sel)[:, 1]

    window_results.append({
        "window": i,
        "n_val": len(y_val),
        "n_test": len(y_test),
        "val_logloss": log_loss(y_val, val_proba),
        "test_logloss": log_loss(y_test, test_proba),
        "test_acc": accuracy_score(y_test, (test_proba > 0.5).astype(int)),
    })

    all_val_true.append(y_val); all_val_proba.append(val_proba)
    all_test_true.append(y_test); all_test_proba.append(test_proba)

# --- Pool everything: concatenate, don't average ---
# Concatenating before scoring naturally weights each window by its
# sample count. Window 6 (252 matches) counts 252x, Window 2 (32) counts
# 32x — no manual weighting needed, unlike averaging six per-window scores.

pooled_val_true = np.concatenate(all_val_true)
pooled_val_proba = np.concatenate(all_val_proba)
pooled_test_true = np.concatenate(all_test_true)
pooled_test_proba = np.concatenate(all_test_proba)

pooled_test_logloss = log_loss(pooled_test_true, pooled_test_proba)
pooled_test_brier = brier_score_loss(pooled_test_true, pooled_test_proba)
pooled_test_acc = accuracy_score(pooled_test_true, (pooled_test_proba > 0.5).astype(int))

print(f"Pooled test log-loss: {pooled_test_logloss:.4f}")
print(f"Pooled test Brier:    {pooled_test_brier:.4f}")
print(f"Pooled test accuracy: {pooled_test_acc:.4f}  (n={len(pooled_test_true)})")

for r in window_results:
    print(r)


# --- Bootstrap CI, so you can tell if an iteration actually moved the needle ---

def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi


mean_ll, lo, hi = bootstrap_logloss_ci(pooled_test_true, pooled_test_proba)
print(f"Pooled log-loss: {mean_ll:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Pooled test log-loss: 0.6605
Pooled test Brier:    0.2335
Pooled test accuracy: 0.5730  (n=452)
{'window': 0, 'n_val': 297, 'n_test': 102, 'val_logloss': 0.6209630425852306, 'test_logloss': 0.6814786872369184, 'test_acc': 0.5294117647058824}
{'window': 1, 'n_val': 198, 'n_test': 48, 'val_logloss': 0.6931366803193373, 'test_logloss': 0.6930103663669729, 'test_acc': 0.5625}
{'window': 2, 'n_val': 375, 'n_test': 72, 'val_logloss': 0.6257737491289108, 'test_logloss': 0.6298446518023798, 'test_acc': 0.5972222222222222}
{'window': 3, 'n_val': 378, 'n_test': 102, 'val_logloss': 0.6137391981820561, 'test_logloss': 0.6612865194097362, 'test_acc': 0.5784313725490197}
{'window': 4, 'n_val': 378, 'n_test': 72, 'val_logloss': 0.6113697895875821, 'test_logloss': 0.6164075379334308, 'test_acc': 0.625}
{'window': 5, 'n_val': 378, 'n_test': 56, 'val_logloss': 0.6446986570699088, 'test_logloss': 0.6889286570586218, 'test_acc': 0.5535714285714286}
Pooled log-loss: 0.6608  95% CI [0.6316, 0.6930]
